# Source B — Home Systems (diagnostic) — projection 2035 — Norte Amazónica

**Rôle diagnostic uniquement**, comme en 2025 (`reality/home_systems.ipynb`) : ce notebook ne
produit **pas** `Demands.csv` (c'est le rôle de `demande.ipynb` dans ce même dossier) et ne
recalcule **pas** les capacités installées `PV_HS`/`HS_DIESEL`/`BATT_HS` — `technologies.ipynb`
les force déjà directement à `f_min=0` à cet horizon (kits légers 2012 hors durée de vie
catalogue).

Seul indicateur recalculé ici : `share_dispersion = B / (A + B)`, en **électricité finale**
(hors cuisson — même convention que le diagnostic 2025), pour les deux jeux de demande
(`no_transition_late_access`, `early_access`) produits par `demande.ipynb`. Sortie informative uniquement :
`output_energyscope_2035/source_B_home_systems_diagnostic.csv` — **hors périmètre
EnergyScope, non copiée vers le dépôt `EnergyScope_BO_nord_amazonia`**.


## 0. Contrôle — `Layers_in_out.csv` vs référence EnergyScope

In [1]:
import pandas as pd

LIO_LOCAL_PATH = "../data/Layers_in_out.csv"
LIO_REFERENCE_PATH = "../../../EnergyScope_BO_nord_amazonia/Data/2025/reality/00_INDEP/Layers_in_out.csv"

lio_local = pd.read_csv(LIO_LOCAL_PATH, sep=";", header=0, index_col=0)
lio_reference = pd.read_csv(LIO_REFERENCE_PATH, sep=";", header=0, index_col=0)

problems = []

only_local_rows = sorted(set(lio_local.index) - set(lio_reference.index))
only_reference_rows = sorted(set(lio_reference.index) - set(lio_local.index))
if only_local_rows:
    problems.append(f"technologies only in local file: {only_local_rows}")
if only_reference_rows:
    problems.append(f"technologies only in EnergyScope reference: {only_reference_rows}")

only_local_cols = sorted(set(lio_local.columns) - set(lio_reference.columns))
only_reference_cols = sorted(set(lio_reference.columns) - set(lio_local.columns))
if only_local_cols:
    problems.append(f"layers only in local file: {only_local_cols}")
if only_reference_cols:
    problems.append(f"layers only in EnergyScope reference: {only_reference_cols}")

common_rows = sorted(set(lio_local.index) & set(lio_reference.index))
common_cols = sorted(set(lio_local.columns) & set(lio_reference.columns))
changed_rows = sorted(
    tech for tech in common_rows
    if not lio_local.loc[tech, common_cols].equals(lio_reference.loc[tech, common_cols])
)
if changed_rows:
    problems.append(f"technologies with different coefficients: {changed_rows}")

if problems:
    raise ValueError(
        f"Layers_in_out.csv differs from the EnergyScope reference ({LIO_REFERENCE_PATH}): "
        + "; ".join(problems)
    )

print(f"OK — Layers_in_out.csv matches the EnergyScope reference ({LIO_REFERENCE_PATH})")


OK — Layers_in_out.csv matches the EnergyScope reference (../../../EnergyScope_BO_nord_amazonia/Data/2025/reality/00_INDEP/Layers_in_out.csv)


## 1. Configuration

In [2]:
import os
import pandas as pd

YEAR = 2035
OUTPUT_DIR = f"output_energyscope_{YEAR}"

CLUSTERS = {
    1: ["Exaltación", "Reyes", "Santa_Rosa_Beni", "Ixiamas"],
    2: ["Bolpebra"],
    3: ["Guayaramerín", "Riberalta", "Puerto_Gonzalo_Moreno"],
    4: ["Bella_Flor", "Filadelfia", "Ingavi", "Nueva_Esperanza", "Porvenir",
        "Puerto_Rico", "San_Lorenzo", "San_Pedro", "Santa_Rosa_Pando",
        "Santos_Mercado", "Sena", "Villa_Nueva"],
    5: ["Cobija"],
}
MUNI_TO_CLUSTER = {m: k for k, munis in CLUSTERS.items() for m in munis}
TRAJ = {"no_transition_late_access": "acces_2050", "early_access": "acces_2035"}

CSVFINAL_TO_RAMP = {
    ("Ixiamas",               "La Paz"): "Ixiamas",
    ("Riberalta",             "Beni"):   "Riberalta",
    ("Guayaramerín",          "Beni"):   "Guayaramerín",
    ("Reyes",                 "Beni"):   "Reyes",
    ("Santa Rosa",            "Beni"):   "Santa_Rosa_Beni",
    ("Exaltación",            "Beni"):   "Exaltación",
    ("Cobija",                "Pando"):  "Cobija",
    ("Porvenir",              "Pando"):  "Porvenir",
    ("Bolpebra",              "Pando"):  "Bolpebra",
    ("Bella Flor",            "Pando"):  "Bella_Flor",
    ("Puerto Rico",           "Pando"):  "Puerto_Rico",
    ("San Pedro",             "Pando"):  "San_Pedro",
    ("Filadelfia",            "Pando"):  "Filadelfia",
    ("Puerto Gonzalo Moreno", "Pando"):  "Puerto_Gonzalo_Moreno",
    ("San Lorenzo",           "Pando"):  "San_Lorenzo",
    ("Sena",                  "Pando"):  "Sena",
    ("Santa Rosa",            "Pando"):  "Santa_Rosa_Pando",
    ("Ingavi",                "Pando"):  "Ingavi",
    ("Nueva Esperanza",       "Pando"):  "Nueva_Esperanza",
    ("Villa Nueva",           "Pando"):  "Villa_Nueva",
    ("Santos Mercado",        "Pando"):  "Santos_Mercado",
}

print(f"YEAR = {YEAR}  ->  OUTPUT_DIR = {OUTPUT_DIR}")


YEAR = 2035  ->  OUTPUT_DIR = output_energyscope_2035


## 2. Source A — électricité finale par cluster (hors cuisson)

Même source que `demande.ipynb` (`projections/output/demande_source_A_projetee.csv`), en
**électricité finale** (pas de conversion en énergie utile ici — `share_dispersion` en 2025 est
défini en électricité finale, cf. `reality/home_systems.ipynb` §4).

In [3]:
source_a_raw = pd.read_csv("../../projections/output/demande_source_A_projetee.csv")
source_a_raw["secteur"] = source_a_raw["secteur"].replace("SERVICES_OTHER", "SERVICES")

def source_a_final_elec_by_cluster(traj):
    sub = source_a_raw[
        (source_a_raw["annee"] == YEAR) & (source_a_raw["trajectoire"] == traj)
        & (source_a_raw["usage_final"] != "COOKING")
    ]
    out = {}
    for cluster_label, group in sub.groupby("cluster"):
        out[int(cluster_label.replace("C", ""))] = group["demande_GWh"].sum()
    return out


## 3. Source B — RAMP sufficiency au prorata, électricité finale (hors cuisson)

Même prorata que `demande.ipynb` (`hh_B(muni, année, trajectoire) / hh_total_muni_2024`),
colonnes RAMP `sufficiency_*` **avant** conversion en énergie utile — `restaurant_kitchen`
(cuisson) exclue, comme en 2025.

In [4]:
split_raw = pd.read_csv("../../projections/output/split_abc_projete.csv")
split_raw["muni_ramp"] = split_raw.apply(
    lambda r: CSVFINAL_TO_RAMP.get((str(r["municipio"]).strip(), str(r["departamento"]).strip())),
    axis=1,
)
unmapped = split_raw[split_raw["muni_ramp"].isna()][["municipio", "departamento"]].drop_duplicates()
if len(unmapped):
    raise ValueError(f"Unmapped municipalities: {unmapped.values.tolist()}")

hh_total_2024 = (
    split_raw[split_raw["annee"] == 2024].drop_duplicates("muni_ramp")
    .set_index("muni_ramp")["hh_total"].to_dict()
)
hh_b_year = (
    split_raw[split_raw["annee"] == YEAR]
    .set_index(["muni_ramp", "trajectoire"])["hh_B"].to_dict()
)

SUFF_RAMP_DIR = "../sufficiency/data ramp"
RAMP_COLS_NO_COOKING_NO_WATERHEAT = None  # filled below from the first file read

_ramp_cache = {}

def _read_ramp_suff(muni):
    if muni not in _ramp_cache:
        path = f"{SUFF_RAMP_DIR}/{muni}/load_curve_energy_service_full_year_Norte_Amazonia.csv"
        if not os.path.exists(path):
            raise FileNotFoundError(f"RAMP sufficiency output missing for '{muni}': {path}")
        _ramp_cache[muni] = pd.read_csv(path)
    return _ramp_cache[muni]


def source_b_final_elec_by_cluster(traj):
    out = {k: 0.0 for k in CLUSTERS}
    for muni, cluster_id in MUNI_TO_CLUSTER.items():
        hh_b = hh_b_year.get((muni, traj), 0.0)
        f = hh_b / hh_total_2024[muni] if hh_total_2024[muni] > 0 else 0.0
        df = _read_ramp_suff(muni)
        cols = [c for c in df.columns if c not in ("time", "restaurant_kitchen")]
        gwh_elec = df[cols].sum().sum() / 60_000_000_000.0 * f
        out[cluster_id] += gwh_elec
    return out


## 4. Diagnostic `share_dispersion` par cluster et par jeu

In [5]:
rows = []
for jeu, traj in TRAJ.items():
    a = source_a_final_elec_by_cluster(traj)
    b = source_b_final_elec_by_cluster(traj)
    print(f"\n{jeu} ({traj}) @ {YEAR} — share_dispersion = B / (A+B), électricité finale hors cuisson:")
    for cluster_id in sorted(CLUSTERS):
        a_gwh = a.get(cluster_id, 0.0)
        b_gwh = b.get(cluster_id, 0.0)
        total = a_gwh + b_gwh
        share = b_gwh / total if total > 0 else 0.0
        rows.append({"jeu": jeu, "trajectoire": traj, "cluster": f"C{cluster_id}",
                     "A_final_elec_GWh": a_gwh, "B_final_elec_GWh": b_gwh,
                     "total_final_elec_GWh": total, "share_dispersion": share})
        print(f"  C{cluster_id}: A={a_gwh:8.4f}  B={b_gwh:7.4f}  total={total:8.4f}  "
              f"share_dispersion={100*share:5.2f}%")

diag = pd.DataFrame(rows)
out_path = f"{OUTPUT_DIR}/source_B_home_systems_diagnostic.csv"
os.makedirs(OUTPUT_DIR, exist_ok=True)
diag.to_csv(out_path, index=False)
print(f"\n-> {out_path}  (diagnostic seulement, PAS copié vers EnergyScope_BO_nord_amazonia)")



no_transition_late_access (acces_2050) @ 2035 — share_dispersion = B / (A+B), électricité finale hors cuisson:
  C1: A= 19.1027  B= 5.3862  total= 24.4889  share_dispersion=21.99%
  C2: A=  1.0867  B= 0.6495  total=  1.7363  share_dispersion=37.41%
  C3: A=112.5675  B= 3.3727  total=115.9401  share_dispersion= 2.91%
  C4: A= 44.9795  B= 6.8694  total= 51.8488  share_dispersion=13.25%
  C5: A= 59.5292  B= 0.1835  total= 59.7126  share_dispersion= 0.31%



early_access (acces_2035) @ 2035 — share_dispersion = B / (A+B), électricité finale hors cuisson:
  C1: A= 26.3945  B= 3.9098  total= 30.3043  share_dispersion=12.90%
  C2: A=  1.8488  B= 0.3575  total=  2.2063  share_dispersion=16.20%
  C3: A=125.8407  B= 0.9627  total=126.8034  share_dispersion= 0.76%
  C4: A= 59.2239  B= 4.5981  total= 63.8220  share_dispersion= 7.20%
  C5: A= 60.4737  B= 0.0000  total= 60.4737  share_dispersion= 0.00%

-> output_energyscope_2035/source_B_home_systems_diagnostic.csv  (diagnostic seulement, PAS copié vers EnergyScope_BO_nord_amazonia)
